# 문제 1-2 : 2단계 Multi Chain 만들기 - 여행지 정보 시스템

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()
PROVIDER = "groq"
CFG = {
    "groq": {"key": os.getenv("GROQ_API_KEY"), "url": "https://api.groq.com/openai/v1", "model": "openai/gpt-oss-120b"},
    "openai": {"key": os.getenv("OPENAI_API_KEY"), "url": None, "model": "gpt-4o-mini"},
}
c = CFG[PROVIDER]
if not c["key"]:
    raise RuntimeError(f".env 파일에 {PROVIDER.upper()}_API_KEY 가 없습니다.")

llm = ChatOpenAI(api_key=c["key"], base_url=c["url"], model=c["model"], temperature=0.7)

p1 = ChatPromptTemplate.from_messages([
    ("system", "당신은 세계 여행 전문가입니다. 사용자가 입력한 여행지의 대표적인 관광 명소를 1가지만 추천하세요. "
               "설명 없이 명소 이름만 한 줄로 답하세요."),
    ("user", "여행지: {city}"),
])

p2 = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 여행 가이드입니다. 아래 형식을 정확히 지켜 한국어로 답하세요.\n\n"
               "명소: <명소 이름>\n\n"
               "역사:\n1. ...\n2. ...\n\n"
               "특징:\n1. ...\n2. ...\n\n"
               "방문 팁:\n1. ...\n2. ..."),
    ("user", "{place}에 대한 상세 정보(역사, 특징, 방문 팁)를 알려주세요."),
])

chain1 = p1 | llm | StrOutputParser()
chain2 = p2 | llm | StrOutputParser()

chain = {"place": chain1} | RunnablePassthrough.assign(info=chain2)

city = input("여행지를 입력하세요 (예: 로마, 뉴욕, 도쿄): ") or "로마"
try:
    r = chain.invoke({"city": city})
    print(f"입력: {city}")
    print(f"\n[1단계 결과] {r['place']}")
    print(f"\n[2단계 결과]\n{r['info']}")
except Exception as e:
    print("오류:", e)


입력: 서울

[1단계 결과] 경복궁

[2단계 결과]
명소: 경복궁

역사:
1. 1395년 조선 태조 이성계가 한양(현재의 서울)으로 천도하면서 건립한 조선 왕조의 첫 번째 궁궐이다.  
2. 1592년 임진왜란으로 대부분이 파괴된 뒤, 1611년부터 1619년 사이에 선조에 의해 재건되었다.  
3. 1910년 일제강점기에는 궁궐이 해체되고 경복궁 근처에 조선총독부가 들어서면서 많은 건축물이 소실되었다.  
4. 1990년대 이후 복원 사업이 진행되어 현재는 500여 년의 역사를 간직한 문화재로 복원·보존되고 있다.

특징:
1. **근정전** – 조선 왕이 즉위식·대외 의전·정치 회의를 진행하던 주요 전각으로, 화려한 단청과 기와 지붕이 인상적이다.  
2. **경회루** – 궁궐 내에서 가장 높은 건축물로, 연못 위에 떠 있는 듯한 모습이 아름다우며, 왕과 왕비가 연회를 열던 장소이다.  
3. **수문장** – 광화문 앞에 위치한 수문장은 조선시대 왕이 외부와 교류할 때 사용했던 관문으로, 전통적인 목조 건축 양식을 보여준다.  
4. **궁궐 정원 및 연못** – 고궁 안의 아름다운 정원과 인왕산을 배경으로 한 연못은 사계절 내내 다른 풍경을 제공한다.

방문 팁:
1. **입장 시간과 티켓** – 오전 9시부터 오후 6시(입장은 5시 30분까지)까지 개방되며, 온라인 사전 예매를 하면 대기 시간을 크게 줄일 수 있다.  
2. **복장과 신발** – 궁궐 내부는 바닥이 돌로 되어 있어 편안한 신발을 착용하고, 날씨에 맞는 옷차림을 준비한다.  
3. **가이드 투어 활용** – 영어·중국어·일본어 등 다국어 오디오 가이드를 대여하면 주요 건물과 역사적 배경을 쉽게 이해할 수 있다.  
4. **주변 연계 관광** – 경복궁을 방문한 뒤 청와대·국립민속박물관·인사동 등을 함께 둘러보면 한 번에 풍부한 문화 체험이 가능하다.
